# Level 3 QML Assignment - Quanvolutional Digits

This assignment is the natural next step after:

1. training a hybrid quantum-classical classifier,
2. fitting a target probability distribution with a QNN.

Here you will use a quantum circuit as a **local feature extractor** for image data. The task is to classify handwritten digits using:

- a raw-pixel classical baseline,
- a fixed quanvolutional feature extractor followed by a classical classifier.

The assignment is connected to `04_quanvolution.ipynb`.


## Goal

Build and evaluate a small quanvolutional pipeline:

$$
8 \times 8 \text{ image}
\rightarrow
2 \times 2 \text{ patches}
\rightarrow
4\text{-qubit quantum filter}
\rightarrow
4 \times 4 \times 4 \text{ feature tensor}
\rightarrow
\text{classical classifier}.
$$

Your final notebook should include:

- a visualized multiclass digit dataset,
- a classical raw-pixel baseline,
- a quanvolutional feature extractor,
- a classifier trained on quanvolutional features,
- accuracy and runtime comparison,
- a short interpretation of whether the quantum feature extractor helped.


## Imports

Use the existing `uv` environment. Do not add install cells.


In [ ]:
# Pseudo-code:
#   1. import numerical, plotting, sklearn, torch, and PennyLane tools
#   2. set random seeds
#   3. define the digit classes for the assignment

# Setup: imports and reproducibility.
import time

import matplotlib.pyplot as plt
import numpy as np
import pennylane as qml
import torch
from sklearn.datasets import load_digits
from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(19)
np.random.seed(19)
torch.manual_seed(19)

digit_classes = np.array([0, 1, 2])
n_classes = len(digit_classes)

print("PennyLane version:", qml.__version__)
print("PyTorch version:", torch.__version__)
print("classes:", digit_classes)


## Dataset

Use the `sklearn` digits dataset. Each image is \(8 \times 8\), which keeps the quantum preprocessing small enough for simulation.

Use only digits 0, 1, and 2 to make this a multiclass task:

$$
y \in \{0,1,2\}.
$$

We will create a small balanced subset so that the quanvolution step runs quickly.


In [ ]:
# Pseudo-code:
#   1. load sklearn digits
#   2. keep classes 0, 1, and 2
#   3. create a balanced train/test subset
#   4. visualize representative images

# Data setup shared by all models.
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits

digits = load_digits()
mask = np.isin(digits.target, digit_classes)
images = digits.images[mask] / 16.0
labels_original = digits.target[mask]
label_map = {digit: index for index, digit in enumerate(digit_classes)}
labels = np.array([label_map[value] for value in labels_original])

train_indices = []
test_indices = []
for class_id in range(n_classes):
    class_indices = np.where(labels == class_id)[0]
    class_indices = rng.permutation(class_indices)
    train_indices.extend(class_indices[:40])
    test_indices.extend(class_indices[40:65])

train_indices = rng.permutation(train_indices)
test_indices = rng.permutation(test_indices)

X_train_images = images[train_indices]
y_train = labels[train_indices]
X_test_images = images[test_indices]
y_test = labels[test_indices]

fig, axes = plt.subplots(3, 5, figsize=(7, 4.5))
for class_id in range(n_classes):
    examples = X_train_images[y_train == class_id][:5]
    for col, image in enumerate(examples):
        axes[class_id, col].imshow(image, cmap="gray", vmin=0, vmax=1)
        axes[class_id, col].axis("off")
        if col == 0:
            axes[class_id, col].set_title(f"digit {digit_classes[class_id]}")
plt.tight_layout()
plt.show()

print("train images:", X_train_images.shape)
print("test images:", X_test_images.shape)


## Warm-Up: One Quantum Patch Filter

A quanvolutional filter takes a local patch and returns classical features from quantum measurements.

For a \(2 \times 2\) patch, use 4 qubits:

$$
[p_0,p_1,p_2,p_3]
\rightarrow
R_Y(\pi p_0) \otimes R_Y(\pi p_1) \otimes R_Y(\pi p_2) \otimes R_Y(\pi p_3).
$$

Then apply a shallow quantum circuit and measure expectation values:

$$
z_j = \langle Z_j \rangle.
$$


In [ ]:
# Pseudo-code:
#   1. create one 2x2 patch from an image
#   2. create a 4-qubit quantum filter
#   3. encode pixels as RY rotations
#   4. measure four Pauli-Z expectations

# TODO: implement a one-patch quantum filter.
#
# Suggested structure:
#
# patch = X_train_images[0, :2, :2]
# filter_weights = ...
# dev = qml.device("default.qubit", wires=4)
#
# @qml.qnode(dev)
# def quantum_filter(features, weights):
#     ...
#     return tuple(qml.expval(qml.PauliZ(wire)) for wire in range(4))
#
# decoded_patch = quantum_filter(patch.ravel(), filter_weights)


## Task 1 - Implement Quanvolution

Write a function that applies your quantum filter to every non-overlapping \(2 \times 2\) patch of an \(8 \times 8\) image.

With patch size 2 and stride 2:

$$
8 \times 8
\rightarrow
4 \times 4 \text{ patches}.
$$

If each patch returns 4 expectation values, the quanvolutional feature tensor has shape:

$$
4 \times 4 \times 4.
$$


In [ ]:
# TODO: implement quanvolution for one image.
#
# Suggested structure:
#
# def quanvolve_image(image, filter_weights):
#     output = np.zeros((4, 4, 4))
#     for row in range(4):
#         for col in range(4):
#             patch = ...
#             output[row, col] = ...
#     return output
#
# quanv_example = quanvolve_image(X_train_images[0], filter_weights)
# print(quanv_example.shape)


## Task 2 - Build Feature Matrices

Create two feature representations:

1. raw pixels:

   $$
   X_{\mathrm{raw}} \in \mathbb{R}^{n \times 64},
   $$

2. quanvolutional features:

   $$
   X_{\mathrm{quanv}} \in \mathbb{R}^{n \times 64}.
   $$

Standardize both using training-set mean and standard deviation.


In [ ]:
# TODO: build raw-pixel and quanvolutional feature matrices.
#
# Suggested structure:
#
# X_train_raw = X_train_images.reshape(len(X_train_images), -1)
# X_test_raw = X_test_images.reshape(len(X_test_images), -1)
#
# X_train_quanv = np.array([...])
# X_test_quanv = np.array([...])
#
# def standardize(train, test):
#     ...
#     return train_scaled, test_scaled


## Task 3 - Train Classifiers

Train the same small PyTorch classifier on both feature sets.

Requirements:

- use `torch.nn.CrossEntropyLoss`,
- output 3 logits,
- report train and test accuracy,
- keep the classifier architecture the same for both feature types.


In [ ]:
# TODO: implement and train the classifier.
#
# Suggested functions:
#
# def train_classifier(X_train, y_train, X_test, y_test):
#     ...
#     return history, train_accuracy, test_accuracy, runtime, predictions
#
# raw_results = train_classifier(...)
# quanv_results = train_classifier(...)


## Task 4 - Compare and Interpret

Produce:

- loss curves for raw-pixel and quanvolutional classifiers,
- a table of train/test accuracy and runtime,
- a confusion matrix for the quanvolutional classifier,
- a short written answer.

In your answer, discuss:

1. Did quanvolution improve test accuracy?
2. How many quantum circuit executions were needed?
3. Is this evidence of quantum advantage?
4. What would you change if the dataset were larger?


In [ ]:
# TODO: visualize and interpret the comparison.
#
# Include:
# - loss curves,
# - accuracy/runtime table,
# - confusion matrix,
# - short written conclusion.


## Submission Checklist

Before submitting, make sure your notebook contains:

- dataset visualization,
- one-patch quantum filter,
- full quanvolution function,
- raw and quanvolutional feature matrices,
- trained classifiers,
- metrics and runtime comparison,
- interpretation of results.
